# 🚀 모델 내보내기 및 배포

이 노트북은 학습된 LoRA 및 OSFT 모델을 내보내고 RHOAI 3.5에 배포합니다.

## 처리 단계

| # | 단계 | 설명 |
|---|------|------|
| 1 | LoRA 내보내기 | 어댑터 저장 + 기본 모델 병합 (merged model) |
| 2 | OSFT 내보내기 | HuggingFace 포맷으로 변환 |
| 3 | 검증 | 리로드 테스트, 해시 체크 |
| 4 | S3 업로드 | S3 호환 스토리지로 전송 |
| 5 | 서빙 매니페스트 | InferenceService YAML 생성 및 적용 |
| 6 | 스모크 테스트 | 채팅 완료 + 도구 호출 테스트 |

### 사전 요구사항
- `03_lora_finetuning.ipynb` 또는 `04_osft_finetuning.ipynb` 완료
- S3 접근 설정 (업로드 시)
- RHOAI 클러스터 접근 (배포 시)

In [ ]:
"""환경 부트스트랩 — local과 workbench 모두 지원."""

import subprocess, sys, os
from pathlib import Path

# 프로젝트 루트 탐색
_nb_dir = Path.cwd()
_project_root = _nb_dir
for _p in [_nb_dir] + list(_nb_dir.parents):
    if (_p / "pyproject.toml").exists():
        _project_root = _p
        break

# 패키지 설치 확인 및 자동 설치
try:
    import rhoai_model_training_lab  # noqa: F401
    print("✅ rhoai_model_training_lab 패키지 확인됨")
except ImportError:
    print("📦 패키지 설치 중... (최초 1회)")
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-e", str(_project_root),
             "--extra-index-url", "https://pypi.org/simple/"],
            stdout=subprocess.DEVNULL,
        )
        print("✅ 설치 완료")
    except subprocess.CalledProcessError:
        # Red Hat Workbench 등 제한된 환경: sys.path fallback
        print("⚠️  pip install 실패 — sys.path fallback 사용")
        _src = str(_project_root / "src")
        if _src not in sys.path:
            sys.path.insert(0, _src)
        # 핵심 의존성만 설치 시도
        for _dep in ["pydantic", "python-dotenv", "pyyaml", "rich", "httpx"]:
            try:
                __import__(_dep.replace("-", "_"))
            except ImportError:
                subprocess.call(
                    [sys.executable, "-m", "pip", "install", _dep,
                     "--extra-index-url", "https://pypi.org/simple/"],
                    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
                )
        import rhoai_model_training_lab  # noqa: F401
        print("✅ sys.path fallback 설정 완료")


In [ ]:
"""Export LoRA adapter and merge to full model."""

import os
import json
import hashlib
from pathlib import Path

from rhoai_model_training_lab.config import (
    load_env, load_training_config, PROJECT_ROOT,
)

load_env()

lora_config = load_training_config("lora")
model_id = lora_config["model"]["model_id"]

adapter_path = PROJECT_ROOT / lora_config["export"]["adapter_path"]
merged_path = PROJECT_ROOT / lora_config["export"]["merged_path"]
checkpoint_dir = PROJECT_ROOT / lora_config["training_args"]["output_dir"]

print("=" * 70)
print("📦 LoRA 모델 내보내기")
print("=" * 70)

try:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    import torch

    # 1. Save adapter separately
    print("\n1️⃣ LoRA 어댑터 저장...")
    adapter_path.mkdir(parents=True, exist_ok=True)

    # Copy adapter files from checkpoint
    adapter_source = checkpoint_dir
    adapter_config_file = adapter_source / "adapter_config.json"
    if not adapter_config_file.exists():
        checkpoints = sorted(checkpoint_dir.glob("checkpoint-*"))
        if checkpoints:
            adapter_source = checkpoints[-1]

    import shutil
    for f in adapter_source.glob("adapter_*"):
        shutil.copy2(f, adapter_path / f.name)
    for f in adapter_source.glob("tokenizer*"):
        shutil.copy2(f, adapter_path / f.name)
    if (adapter_source / "special_tokens_map.json").exists():
        shutil.copy2(adapter_source / "special_tokens_map.json", adapter_path)

    print(f"  ✅ 어댑터 저장 완료: {adapter_path}")

    # 2. Merge adapter with base model
    if lora_config["export"].get("save_merged", True):
        print("\n2️⃣ 기본 모델과 어댑터 병합 중...")
        print("  (이 과정은 몇 분이 소요될 수 있습니다)")

        base_model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.bfloat16,
            trust_remote_code=True,
            device_map="auto",
        )
        model = PeftModel.from_pretrained(base_model, str(adapter_source))
        merged_model = model.merge_and_unload()

        merged_path.mkdir(parents=True, exist_ok=True)
        merged_model.save_pretrained(str(merged_path))

        tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
        tokenizer.save_pretrained(str(merged_path))

        print(f"  ✅ 병합 모델 저장 완료: {merged_path}")
        del base_model, model, merged_model
        torch.cuda.empty_cache()
    else:
        print("  ⏭️ 병합 건너뜀 (설정에서 비활성화)")

except Exception as exc:
    print(f"❌ LoRA 내보내기 실패: {exc}")
    print("  GPU 메모리가 부족하면 병합을 별도 프로세스에서 실행하세요.")
    raise

In [ ]:
"""Export OSFT model to HuggingFace format."""

osft_config = load_training_config("osft")
osft_checkpoint = PROJECT_ROOT / osft_config["training_args"]["output_dir"]
osft_export_path = PROJECT_ROOT / osft_config["export"]["output_path"]

print("=" * 70)
print("📦 OSFT 모델 내보내기")
print("=" * 70)

try:
    from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
    import torch

    print(f"\nOSFT 체크포인트: {osft_checkpoint}")
    print(f"내보내기 경로: {osft_export_path}")

    # OSFT exports the full model (not an adapter)
    print("\n모델 로드 및 HuggingFace 포맷 변환 중...")
    osft_model = AutoModelForCausalLM.from_pretrained(
        str(osft_checkpoint),
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
        device_map="auto",
    )

    osft_export_path.mkdir(parents=True, exist_ok=True)
    osft_model.save_pretrained(str(osft_export_path))

    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    tokenizer.save_pretrained(str(osft_export_path))

    print(f"✅ OSFT 모델 내보내기 완료: {osft_export_path}")

    del osft_model
    torch.cuda.empty_cache()

except Exception as exc:
    print(f"❌ OSFT 내보내기 실패: {exc}")
    raise

In [ ]:
"""Validate exported models — reload test and hash check."""

from rhoai_model_training_lab.data import compute_file_checksum

print("=" * 70)
print("🔍 내보내기 검증")
print("=" * 70)

validation_results = {}

# Validate LoRA merged model
for name, model_path in [("LoRA (병합)", merged_path), ("OSFT", osft_export_path)]:
    print(f"\n--- {name} 검증 ---")
    checks = {}

    # Check required files
    required = ["config.json", "tokenizer_config.json"]
    for req in required:
        exists = (model_path / req).exists()
        checks[req] = exists
        print(f"  {'✅' if exists else '❌'} {req}")

    # Check weight files
    safetensors = list(model_path.glob("*.safetensors"))
    bin_files = list(model_path.glob("*.bin"))
    weight_files = safetensors or bin_files
    checks["weights"] = bool(weight_files)
    print(f"  {'✅' if weight_files else '❌'} 가중치 파일: {len(weight_files)}개")

    # Check index file
    index = (
        (model_path / "model.safetensors.index.json").exists()
        or (model_path / "pytorch_model.bin.index.json").exists()
        or len(weight_files) == 1
    )
    checks["index"] = index
    print(f"  {'✅' if index else '⚠️ '} 인덱스 파일")

    # Compute hash of weight files
    if weight_files:
        h = hashlib.sha256()
        for wf in sorted(weight_files):
            h.update(compute_file_checksum(wf).encode())
        model_hash = h.hexdigest()[:16]
        print(f"  📎 모델 해시: {model_hash}")

    # Reload test
    try:
        config = AutoConfig.from_pretrained(str(model_path), trust_remote_code=True)
        tok = AutoTokenizer.from_pretrained(str(model_path), trust_remote_code=True)
        checks["reload"] = True
        print(f"  ✅ 리로드 테스트 통과 ({config.model_type})")
    except Exception as exc:
        checks["reload"] = False
        print(f"  ❌ 리로드 실패: {exc}")

    # Total size
    total_bytes = sum(f.stat().st_size for f in model_path.rglob("*") if f.is_file())
    print(f"  📦 총 크기: {total_bytes / (1024**3):.2f} GB")

    validation_results[name] = all(checks.values())

all_valid = all(validation_results.values())
print(f"\n{'✅ 모든 모델 검증 통과' if all_valid else '❌ 일부 검증 실패'}")

In [ ]:
"""Upload exported models to S3."""

s3_endpoint = os.environ.get("S3_ENDPOINT", "")
s3_bucket = os.environ.get("S3_BUCKET", "rhoai-model-training-lab")

print("=" * 70)
print("☁️  S3 업로드")
print("=" * 70)

if not s3_endpoint:
    print("⚠️  S3_ENDPOINT가 설정되지 않았습니다.")
    print("   S3 업로드를 건너뜁니다.")
    print("   수동 업로드 명령어:")
    print(f"   aws s3 sync {merged_path} s3://{s3_bucket}/models/lora-merged/")
    print(f"   aws s3 sync {osft_export_path} s3://{s3_bucket}/models/osft-exported/")
else:
    try:
        import boto3

        s3 = boto3.client(
            "s3",
            endpoint_url=s3_endpoint,
            aws_access_key_id=os.environ.get("S3_ACCESS_KEY_ID"),
            aws_secret_access_key=os.environ.get("S3_SECRET_ACCESS_KEY"),
            region_name=os.environ.get("S3_REGION", "us-east-1"),
        )

        uploads = [
            (merged_path, "models/lora-merged"),
            (osft_export_path, "models/osft-exported"),
            (adapter_path, "models/lora-adapter"),
        ]

        for local_path, s3_prefix in uploads:
            if not local_path.exists():
                print(f"  ⏭️ {local_path.name} 없음, 건너뜀")
                continue
            print(f"\n  📤 {local_path.name} → s3://{s3_bucket}/{s3_prefix}/")
            for file_path in local_path.rglob("*"):
                if file_path.is_file():
                    rel = file_path.relative_to(local_path)
                    s3_key = f"{s3_prefix}/{rel}"
                    s3.upload_file(str(file_path), s3_bucket, s3_key)
            print(f"  ✅ 업로드 완료")

    except ImportError:
        print("❌ boto3가 설치되지 않았습니다. pip install boto3")
    except Exception as exc:
        print(f"❌ S3 업로드 실패: {exc}")

In [ ]:
"""Render and apply serving manifests (InferenceService)."""

from jinja2 import Template

print("=" * 70)
print("📋 서빙 매니페스트 생성")
print("=" * 70)

INFERENCE_SERVICE_TEMPLATE = """apiVersion: serving.kserve.io/v1beta1
kind: InferenceService
metadata:
  name: {{ name }}
  namespace: {{ namespace }}
  annotations:
    serving.kserve.io/deploymentMode: RawDeployment
spec:
  predictor:
    model:
      modelFormat:
        name: vLLM
      runtime: vllm-runtime
      storageUri: {{ storage_uri }}
      resources:
        limits:
          nvidia.com/gpu: "1"
        requests:
          cpu: "4"
          memory: "16Gi"
      env:
        - name: MODEL_NAME
          value: "{{ model_name }}"
        - name: MAX_MODEL_LEN
          value: "4096"
        - name: TOOL_PARSER
          value: "{{ tool_parser }}"
"""

template = Template(INFERENCE_SERVICE_TEMPLATE)
namespace = os.environ.get("NAMESPACE", "rhoai-lab")

manifests = [
    {
        "name": "lora-model-predictor",
        "model_name": "Qwen3-4B-Instruct-2507-lora",
        "storage_uri": f"s3://{s3_bucket}/models/lora-merged",
    },
    {
        "name": "osft-model-predictor",
        "model_name": "Qwen3-4B-Instruct-2507-osft",
        "storage_uri": f"s3://{s3_bucket}/models/osft-exported",
    },
]

manifest_dir = PROJECT_ROOT / "deploy" / "manifests"
manifest_dir.mkdir(parents=True, exist_ok=True)

for m in manifests:
    rendered = template.render(
        name=m["name"],
        namespace=namespace,
        storage_uri=m["storage_uri"],
        model_name=m["model_name"],
        tool_parser="hermes",
    )
    manifest_file = manifest_dir / f"{m['name']}.yaml"
    manifest_file.write_text(rendered)
    print(f"\n✅ 매니페스트 생성: {manifest_file}")
    print(rendered)

print("\n📝 배포 명령어:")
for m in manifests:
    print(f"  oc apply -f deploy/manifests/{m['name']}.yaml")
print(f"\n  oc get inferenceservice -n {namespace}")

In [ ]:
"""Smoke test — chat completion and tool call test."""

from rhoai_model_training_lab.config import load_yaml_config

print("=" * 70)
print("🧪 서빙 스모크 테스트")
print("=" * 70)

# Check configured endpoints
endpoints_config_path = PROJECT_ROOT / "configs" / "endpoints.example.yaml"
try:
    endpoints_config = load_yaml_config(endpoints_config_path)
except FileNotFoundError:
    endpoints_config = {"endpoints": {}}

test_endpoints = {
    "base": os.environ.get("BASE_SERVING_ENDPOINT", ""),
    "lora": os.environ.get("LORA_SERVING_ENDPOINT", ""),
    "osft": os.environ.get("OSFT_SERVING_ENDPOINT", ""),
}

import httpx

for variant, endpoint in test_endpoints.items():
    if not endpoint:
        print(f"\n⏭️ {variant}: 엔드포인트 미설정, 건너뜀")
        continue

    print(f"\n--- {variant} 모델 테스트 ---")
    print(f"엔드포인트: {endpoint}")

    headers = {}
    token = os.environ.get("SERVING_TOKEN", "")
    if token:
        headers["Authorization"] = f"Bearer {token}"

    ca_bundle = os.environ.get("SERVING_CA_BUNDLE", "")
    verify = ca_bundle if ca_bundle else True

    # 1. Chat completion test
    print("\n  1️⃣ 채팅 완료 테스트")
    try:
        resp = httpx.post(
            f"{endpoint}/chat/completions",
            json={
                "model": endpoints_config.get("endpoints", {}).get(variant, {}).get("model_name", variant),
                "messages": [
                    {"role": "system", "content": "You are a banking support assistant."},
                    {"role": "user", "content": "What is the maximum daily transfer limit for premium accounts?"},
                ],
                "max_tokens": 256,
                "temperature": 0.1,
            },
            headers=headers,
            verify=verify,
            timeout=60,
        )
        resp.raise_for_status()
        result = resp.json()
        content = result["choices"][0]["message"]["content"]
        print(f"  ✅ 응답: {content[:200]}")
    except Exception as exc:
        print(f"  ❌ 실패: {exc}")

    # 2. Tool call test
    print("\n  2️⃣ 도구 호출 테스트")
    try:
        resp = httpx.post(
            f"{endpoint}/chat/completions",
            json={
                "model": endpoints_config.get("endpoints", {}).get(variant, {}).get("model_name", variant),
                "messages": [
                    {"role": "system", "content": "You are a banking assistant. Use tools when needed."},
                    {"role": "user", "content": "Please check the balance of account ACC-12345."},
                ],
                "tools": [{
                    "type": "function",
                    "function": {
                        "name": "get_account_balance",
                        "description": "Get the balance of a bank account",
                        "parameters": {
                            "type": "object",
                            "properties": {
                                "account_id": {"type": "string", "description": "Account ID"}
                            },
                            "required": ["account_id"],
                        },
                    },
                }],
                "max_tokens": 256,
                "temperature": 0.1,
            },
            headers=headers,
            verify=verify,
            timeout=60,
        )
        resp.raise_for_status()
        result = resp.json()
        msg = result["choices"][0]["message"]
        if msg.get("tool_calls"):
            tc = msg["tool_calls"][0]
            print(f"  ✅ 도구 호출: {tc['function']['name']}({tc['function']['arguments']})")
        else:
            print(f"  ⚠️  도구 호출 없음. 응답: {msg.get('content', '')[:200]}")
    except Exception as exc:
        print(f"  ❌ 실패: {exc}")

print("\n" + "=" * 70)
print("배포 완료! 다음 단계:")
print("  📓 06_rag_harness.ipynb — RAG 하네스 구축")
print("  📓 07_evaluate.ipynb — 평가 실행")